In [2]:
pip install twelvedata

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from twelvedata import TDClient

td = TDClient(apikey="0b64d2b2eb1a4e12bcb346bef059df69")

# Download data per year, limit output of 5000 in the free API
rangos = [
    ("2022-01-01", "2022-12-31"),
    ("2023-01-01", "2023-12-31"),
    ("2024-01-01", "2024-12-31"),
    ("2025-01-01", "2025-12-31"),
]

dfs = []
for start, end in rangos:
    ts = td.time_series(
        symbol="DAX",
        interval="1h",
        start_date=start,
        end_date=end,
        outputsize=5000
    )
    dfs.append(ts.as_pandas())

df = pd.concat(dfs)

# DATA PROCESSING

# Reset index to work with with datetime as column
df = df.reset_index()
df.columns = df.columns.str.lower()  # name'datetime'

# Prder per date
df = df.sort_values(by='datetime')

# Round hours to XX:00
df['datetime'] = df['datetime'].dt.ceil('h')

# Delete duplicates after round values
df = df.drop_duplicates(subset='datetime', keep='last')

# Fix datetime like index
df = df.set_index('datetime')

# Include every hour, include those with no values due holidays
full_range = pd.date_range(start='2022-01-01 00:00', end='2025-12-31 23:00', freq='h')
df = df.reindex(full_range)

# Insert NaNs
df = df.ffill().bfill()

# Save
df.to_csv("DAX_hourly_2025.csv")
print(f"Filas: {df.shape[0]}")
print(f"Desde: {df.index.min()} — Hasta: {df.index.max()}")
print(df.head())

Filas: 35064
Desde: 2022-01-01 00:00:00 — Hasta: 2025-12-31 23:00:00
                      open   high    low  close   volume
2022-01-01 00:00:00  32.47  32.57  32.45   32.5  14773.0
2022-01-01 01:00:00  32.47  32.57  32.45   32.5  14773.0
2022-01-01 02:00:00  32.47  32.57  32.45   32.5  14773.0
2022-01-01 03:00:00  32.47  32.57  32.45   32.5  14773.0
2022-01-01 04:00:00  32.47  32.57  32.45   32.5  14773.0
